# BioPerformance SHAP Analysis Notebook

**Objective:** Demonstrate feature importance (global) and per-prediction explainability (local)
for the XGBoost readiness model, as required by the Brief (Section 8).

### Methodology
- Model: XGBoost Regressor (n_estimators=100, max_depth=4, learning_rate=0.1)
- Explainability: SHAP TreeExplainer
- Data: Synthetic athlete data (5 athletes, 90 days each)
- Train/Test: 80/20 time-based split (no leakage)

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

In [ ]:
# Load processed features (must run src/feature_engineering.py first)
# Re-run src/feature_engineering.py --dataset pmdata to switch to real data
df = pd.read_csv('data/processed/features.csv', parse_dates=['date'])
print(f"Loaded {len(df)} rows.")
df.head(2)

In [ ]:
# Prepare data - drop raw fields and targets, keep only engineered features
DROP_COLS = ['date', 'athlete_id', 'target_hooper_tomorrow', 
             'target_fatigue_tomorrow', 'target_soreness_tomorrow', 
             'target_mood_tomorrow', 'target_sleep_quality_tomorrow',
             'fatigue', 'soreness', 'mood', 'sleep_quality', 'srpe']

df_sorted = df.sort_values('date').reset_index(drop=True)
X = df_sorted.drop(columns=DROP_COLS, errors='ignore')
y = df_sorted['target_hooper_tomorrow']

# 80/20 time-based split
split_idx = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")
print(f"Feature count: {X.shape[1]}")

In [ ]:
# Train XGBoost model
model = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = np.mean(np.abs(preds - y_test))
print(f"Model MAE: {mae:.2f} Hooper Index points")
print("Model trained.")

---
## 1. Global Feature Importance (Beeswarm Plot)

This plot shows which features matter most **across all predictions** in the test set.
Features are ranked by their mean absolute SHAP value (top = most important).
Color indicates the feature value (red = high, blue = low).

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

# Beeswarm plot
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, show=False, max_display=15)
plt.tight_layout()
plt.savefig('data/processed/shap_beeswarm_global.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/processed/shap_beeswarm_global.png")

---
## 2. Partial Dependence Plot (PDP)

Shows how the top feature affects predictions on average. Per Taber et al. (2024) methodology - PDP alongside SHAP.


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top_feature = X_test.columns[np.argmax(np.abs(shap_values.values).mean(axis=0))]
print(f"Top SHAP feature: {top_feature}")

fig, ax = plt.subplots(figsize=(8, 4))
PartialDependenceDisplay.from_estimator(model, X_test, [top_feature], ax=ax)
plt.tight_layout()
plt.savefig('data/processed/pdp_top_feature.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/processed/pdp_top_feature.png")


---
## 3. Local Explainability (Waterfall Plot)

This breaks down **a single prediction** into its component SHAP values.
Each feature pushes the prediction up (red) or down (blue) from the baseline.

In [ ]:
# Analyze the first prediction in the test set
instance_idx = 0
print(f"Actual Target (Tomorrow's Hooper): {y_test.iloc[instance_idx]:.2f}")
print(f"Model Prediction: {model.predict(X_test.iloc[[instance_idx]])[0]:.2f}")

plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values[instance_idx], show=False, max_display=10)
plt.tight_layout()
plt.savefig('data/processed/shap_waterfall_local.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/processed/shap_waterfall_local.png")

---
## 4. Top 3 SHAP Drivers (LLM Synthesis Input)

These are the features that most influenced this specific prediction.
This data feeds directly into the Claude coaching insight prompt.

In [ ]:
instance_shap = shap_values[instance_idx]

shap_df = pd.DataFrame({
    'feature_value': instance_shap.data,
    'shap_value': instance_shap.values
}, index=X_test.columns)

shap_df['abs_shap'] = np.abs(shap_df['shap_value'])
top_drivers = shap_df.sort_values('abs_shap', ascending=False).head(3)

print("--- Top 3 SHAP Drivers ---")
for feature_name, row in top_drivers.iterrows():
    if row['shap_value'] > 0:
        effect = f"increased predicted Hooper by {abs(row['shap_value']):.2f} pts (worse readiness)"
    else:
        effect = f"decreased predicted Hooper by {abs(row['shap_value']):.2f} pts (better readiness)"
    print(f"- {feature_name} (={row['feature_value']:.1f}) {effect}")

---
## Summary

This notebook demonstrates:
- **Global explainability** (beeswarm + PDP): Which features drive readiness across all athletes
- **Local explainability** (waterfall): Why a specific prediction was made
- **LLM-ready output**: Top 3 drivers formatted for coach-facing insights

The SHAP values are fully auditable - every prediction can be traced back to its contributing features.